# Example 3: 使用已有的INP来执行

这个例子和 `02_BatchParameterizedJob` 系列的区别**不在于能不能处理 `*INCLUDE`**（两者共用同一套
include 树解析), 而在于**一个批次里变的是什么**:

| 批次形态 | 变的是 | 生成器 | kind | base_spec 里该留空的字段 |
| --- | --- | --- | --- | --- |
| 1 个模板 × N 组参数 | **参数** | `generate_from_array` | `inp_based` | `params` |
| N 个已写好的 deck | **文件** | `generate_from_inp_files` | `existing_inp` | `source_path`、`params` |

底层是同一个 `InpPreparationStrategy`, 流水线都是 `代入参数 → 校验 → 落盘`;
`existing_inp` 就是"参数集为空"的那一档, 额外多一条**断言**: deck 里若还残留 `{{placeholder}}`,
它会告诉你"你把一个模板混进了成品批次", 而不是含糊地说"缺参数"。

**生成器负责填的字段, base_spec 里就别填。** `generate_from_inp_files` 逐文件提供
`kind`、`source_path` 和 `params`, 所以这里的 `source_path` 直接留空, 没必要硬塞一个不相干的
INP 进去。真填了也不会被无声丢弃——生成器会**一次性**(而不是每个文件一条)警告你哪些值被丢掉了。

`options` 会被继承; `meta` 是**合并**而非替换, 你自己的键会和 `source_inp` 共存。

反过来, 不经过生成器的 spec（手写单个 job, 或 `05_DryRunPreview` 里直接喂给 `dry_run("stage")`
的那种）, `source_path` 就是必需的; 留空会在准备阶段报错, 并同时告诉你这两条出路。

In [1]:
import os
import numpy as np

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec
from ABQflow import generate_from_inp_files, degenerate_from_array

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()

In [2]:
base_spec = JobSpec(
	job_name = "existing_inp_example",		# naming='stem' 时不用到; naming='indexed' 时作前缀
	workflow = "modular",
	preparation = PreparationSpec(
		kind = "existing_inp",
		source_path = "",					# 由 generate_from_inp_files 逐文件覆盖
		options = {
			# 未知的 option key 现在会直接抛错, 不再被静默忽略
			'resolve_includes': True		# 走 *INCLUDE 树, 改写成绝对路径
		}
	),
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)

# 每个 scenario 文件各自 *INCLUDE 了 ../planar_stress_main.inp;
# 它是静态的, 于是被绝对路径指向, 两个 job 共用同一份, 不复制。
specs = generate_from_inp_files("./examples/cae_file/scenarios/*.inp", base_spec)

import pprint
pprint.pprint(specs)

[JobSpec(job_name='planar_stress_scenario_1',
         workflow='modular',
         preparation=PreparationSpec(kind='existing_inp',
                                     source_path='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples\\cae_file\\scenarios\\planar_stress_scenario_1.inp',
                                     params={},
                                     options={'resolve_includes': True}),
         preflight=None,
         monolithic_script=None,
         monolithic_params={},
         pre_extraction=[],
         post_extraction=[HookSpec(script_path='./examples/extraction_scripts/get_max_stress_mises.py',
                                   tasks=[{'result_name': 'max_stress_mises'},
                                          {'result_name': 'max_displacement'}])],
         subroutine=None,
         meta={'source_inp': 'c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples\\cae_file\\scenarios\\planar_stress_scenario_1.inp'}),
 JobSpec(job_name='planar_stress_scenario_2',
   

In [3]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = os.path.join(CWD, "examples/03_ExistingInpBatchJob/output"),
	cpus_per_job = 4,
	duplicate_mode = "overwrite",
	abaqus_exe = ABAQUS_CAE,
)

In [4]:
outcomes = processor.run_batch(
	num_parallel_jobs = 2,
)

Output()

In [5]:
outcomes

[JobOutcome(job_name='planar_stress_scenario_1', status='COMPLETED', results={'max_stress_mises': 4525.26025390625, 'max_displacement': 4.189039707183838}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/03_ExistingInpBatchJob/output\\planar_stress_scenario_1', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1788594415.2124164, 'ended_at': 1788594415.215413, 'duration_s': 0.002996683120727539, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1788594415.215413, 'ended_at': 1788594467.356155, 'duration_s': 52.14074182510376, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788594467.356155, 'ended_at': 1788594468.9965413, 'duration_s': 1.6403863430023193, 'error': None}], duration_s=53.786128282547),
 JobOutcome(job_name='planar_stress_scenario_2', status='COMPLETED', results={'max_stress_mises': 6787.890625, 'max_displacement': 5.984

In [6]:
Y = degenerate_from_array(
	outcomes = outcomes,
	output_names = ["max_stress_mises", "max_displacement"],
)
print(f"output: {Y}")

output: [[4.52526025e+03 4.18903971e+00]
 [6.78789062e+03 5.98434210e+00]]
